# PHASE 2: EXECUTION PIPELINE
## Thực thi xử lý dữ liệu chính thức

**Mục tiêu:**
1. Load merged data từ PHASE 0
2. Load configuration từ `final_config.yaml` (đã được review ở PHASE 1.5)
3. Áp dụng pipeline xử lý:
   - Log Transform (nếu config enabled)
   - Outlier Handling (IQR hoặc Isolation Forest)
   - Scaling & Normalization
   - KNN Imputation
4. Lưu output:
   - `dataset_final.csv` - Dữ liệu đã xử lý
   - `scaler_model.pkl` - Scaler model để dùng trên data mới
   - `processing_metadata.json` - Metadata về transformations

In [ ]:
import sys
import os
from pathlib import Path

# Add modules to path
sys.path.insert(0, str(Path.cwd() / 'modules'))

import pandas as pd
import numpy as np
import logging
import yaml
import json

from processing import DataProcessor
from config_handler import ConfigHandler

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Modules imported successfully")

## Step 1: Load merged data

In [ ]:
input_path = './outputs/dataset_merged.csv'

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Merged dataset not found. Run PHASE 0 first: {input_path}")

df = pd.read_csv(input_path)

print(f"✓ Loaded {input_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()[:5]}... ({len(df.columns)} total)")

## Step 2: Load configuration

In [ ]:
config_path = './outputs/final_config.yaml'

# If final_config.yaml doesn't exist, use draft_config.yaml
if not os.path.exists(config_path):
    draft_path = './outputs/draft_config.yaml'
    if os.path.exists(draft_path):
        print(f"final_config.yaml not found, using draft_config.yaml")
        config_path = draft_path
    else:
        raise FileNotFoundError(f"No config file found. Run PHASE 1 first.")

# Load config
config_handler = ConfigHandler()
config = config_handler.load_config(config_path)

print(f"✓ Loaded config from {config_path}")

# Validate config
is_valid, errors = config_handler.validate_config()
if not is_valid:
    print("⚠️  Config validation errors:")
    for error in errors:
        print(f"  - {error}")
else:
    print("✓ Config validation passed")

## Step 3: Display configuration

In [ ]:
print("\n" + "="*50)
print("PIPELINE CONFIGURATION")
print("="*50)

print("\nPhase 1 - Diagnostics:")
log_transform = config.get('phase1', {}).get('log_transform', {})
print(f"  Log Transform Enabled: {log_transform.get('enabled', False)}")
print(f"    - add_constant: {log_transform.get('add_constant', 1.0)}")

outlier_cfg = config.get('phase1', {}).get('outlier_detection', {})
print(f"  Outlier Detection: {outlier_cfg.get('method', 'iqr')}")
print(f"    - action: {outlier_cfg.get('action', 'clip')}")

print("\nPhase 2 - Processing:")
scaling = config.get('phase2', {}).get('scaling', {})
print(f"  Scaling Method: {scaling.get('method', 'standard')}")

imputation = config.get('phase2', {}).get('imputation', {})
print(f"  Imputation: {imputation.get('method', 'knn')}")
print(f"    - n_neighbors: {imputation.get('n_neighbors', 5)}")

## Step 4: Initialize DataProcessor

In [ ]:
# Create processor
processor = DataProcessor(df)

print(f"✓ DataProcessor initialized")
print(f"  Numeric columns: {len(processor.numeric_cols)}")
print(f"  Data shape: {processor.df.shape}")

## Step 5: Run processing pipeline

In [ ]:
# Build processing config
processing_config = {
    'log_transform': config.get('phase1', {}).get('log_transform', {}),
    'outlier_handling': config.get('phase1', {}).get('outlier_detection', {}),
    'scaling': config.get('phase2', {}).get('scaling', {}),
    'imputation': config.get('phase2', {}).get('imputation', {}),
}

# Run processing pipeline
processed_df, scaler = processor.run_processing_pipeline(processing_config)

print("\n" + "="*50)
print("PROCESSING COMPLETE")
print("="*50)

## Step 6: Inspect processed data

In [ ]:
print("\nProcessed Data Summary:")
print(f"  Shape: {processed_df.shape}")
print(f"  Missing values: {processed_df.isna().sum().sum()}")
print(f"  Data types: {processed_df.dtypes.value_counts().to_dict()}")

print("\nFirst few rows:")
print(processed_df.head())

In [ ]:
# Data statistics after processing
print("\n" + "="*50)
print("PROCESSED DATA STATISTICS")
print("="*50)

numeric_cols = processed_df.select_dtypes(include=[np.number]).columns
print("\nMean values (sample):")
print(processed_df[numeric_cols].mean().head())

print("\nStd values (sample):")
print(processed_df[numeric_cols].std().head())

## Step 7: Save outputs

In [ ]:
output_folder = Path('./outputs')
output_folder.mkdir(exist_ok=True)

# Save processed data
output_csv = output_folder / 'dataset_final.csv'
processed_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"✓ Saved processed data: {output_csv}")
print(f"  File size: {output_csv.stat().st_size / (1024*1024):.2f} MB")
print(f"  Shape: {processed_df.shape}")

# Save scaler model
scaler_path = output_folder / 'scaler_model.pkl'
processor.save_scaler(str(scaler_path))
print(f"\n✓ Saved scaler model: {scaler_path}")

## Step 8: Save processing metadata

In [ ]:
# Get and save metadata
metadata = processor.get_metadata()

metadata_path = output_folder / 'processing_metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Saved processing metadata: {metadata_path}")

print("\nProcessing Transformations Applied:")
for i, transform in enumerate(metadata.get('transformations', []), 1):
    print(f"  {i}. {transform.get('type')}")
    if 'method' in transform:
        print(f"     Method: {transform['method']}")

## Final Summary & Checkpoints

In [ ]:
print("\n" + "="*60)
print("🎉 PIPELINE EXECUTION COMPLETE")
print("="*60)

print("\nPhase Progression:")
print("  ✓ PHASE 0: Data Integration - dataset_merged.csv")
print("  ✓ PHASE 1: Auto Profiling - Diagnostics reports")
print("  ✓ PHASE 1.5: Config Review - final_config.yaml")
print("  ✓ PHASE 2: Execution Pipeline - Final processing")

print("\nOutput Files Generated:")
output_files = [
    'dataset_merged.csv',
    'diagnostic_report.csv',
    'diagnostic_report.json',
    'boxplots.png',
    'histograms.png',
    'draft_config.yaml',
    'final_config.yaml',
    'dataset_final.csv',
    'scaler_model.pkl',
    'processing_metadata.json',
]

for file in output_files:
    path = output_folder / file
    if path.exists():
        size = path.stat().st_size
        if size > 1024*1024:
            size_str = f"{size / (1024*1024):.2f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size} B"
        print(f"  ✓ {file} ({size_str})")
    else:
        print(f"  ⚠ {file} (not found)")

print("\nKey Artifacts:")
print(f"  📊 Input: {input_path}")
print(f"  ⚙️  Config: {config_path}")
print(f"  📈 Output: {output_csv}")
print(f"  🔧 Model: {scaler_path}")

print("\n" + "="*60)